# 04 — GLM: Binary Relevance (Production Model)

Five independent binary logistic regressions, one per fraud label.
This is the production classifier.

**Also covers:** proportional-odds (severity), Poisson (velocity), power-set multinomial (companion).

**Key outputs:**
- Per-label coefficient tables with odds ratios and 95% CIs
- Wald test significance flags
- Hosmer-Lemeshow calibration per label
- Deviance and Pearson residual plots
- VIF for multicollinearity audit
- Baseline log-likelihoods (needed for LR-test gate in 06/07)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS
from src.features import build_features, label_matrix, severity_vector, velocity_count_vector
from src.models.glm import BinaryRelevanceGLM, ProportionalOddsModel, PoissonVelocity, PowerSetMNL
from src.evaluation import multi_label_report, hosmer_lemeshow, lr_test

In [ ]:
train = pd.read_parquet('../data/processed/train_labeled.parquet')
test  = pd.read_parquet('../data/processed/test_labeled.parquet')

X_train = build_features(train)
X_test  = build_features(test)
y_train = label_matrix(train)
y_test  = label_matrix(test)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')

In [ ]:
# VIF audit before fitting
vif = pd.DataFrame({
    'feature': X_train.columns,
    'VIF': [variance_inflation_factor(X_train.values, i)
            for i in range(X_train.shape[1])]
})
print('High VIF features (>5):')
print(vif[vif['VIF'] > 5].sort_values('VIF', ascending=False).to_string(index=False))

In [ ]:
# Fit binary relevance model
br = BinaryRelevanceGLM(maxiter=300)
br.fit(X_train, y_train)
print(br.summary())

In [ ]:
# Odds ratios for each label
for label in LABEL_COLS:
    print(f'\n── {label} ──')
    or_df = br.odds_ratios(label)
    significant = or_df[or_df['pvalue'] < 0.05]
    print(significant[['OR','OR_ci_lo','OR_ci_hi','pvalue']].round(4))

In [ ]:
# Hosmer-Lemeshow calibration per label
proba = br.predict_proba(X_test)
for label in LABEL_COLS:
    hl = hosmer_lemeshow(y_test[label].values, proba[label].values)
    status = 'PASS' if hl['p_value'] > 0.05 else 'FAIL'
    print(f'{label}: HL={hl["HL"]:.2f}  p={hl["p_value"]:.4f}  [{status}]')

In [ ]:
# Multi-label evaluation
y_score = proba.values
y_pred  = (y_score >= 0.5).astype(int)
report  = multi_label_report(y_test.values, y_pred, y_score)

print(f"Hamming loss:      {report['hamming_loss']:.4f}")
print(f"Subset accuracy:   {report['subset_accuracy']:.4f}")
print(f"Label ranking AP:  {report['label_ranking_ap']:.4f}")
print(f"Mean AUC:          {report['mean_auc']:.4f}")
print('\nPer-label AUC:')
for lbl, d in report['per_label'].items():
    print(f"  {lbl}: AUC={d.get('auc', 'N/A'):.4f}  AP={d.get('ap', 'N/A'):.4f}")

In [ ]:
# Save baseline log-likelihoods for LR-test gate in later notebooks
import json
lls = br.log_likelihoods()
with open('../data/processed/baseline_log_likelihoods.json', 'w') as f:
    json.dump(lls, f, indent=2)
print('Saved baseline log-likelihoods:', lls)

In [ ]:
# Proportional-odds model for severity
sev_train = severity_vector(train)
sev_test  = severity_vector(test)
po = ProportionalOddsModel()
po.fit(X_train, sev_train)
print(po.result.summary())

In [ ]:
# Poisson velocity model
V_train = velocity_count_vector(train)
pois = PoissonVelocity()
pois.fit(X_train, V_train)
print(f'Model type: {pois.model_type}')
print(pois.result.summary())